# Convert a processed UMI Zarr replay buffer to LeRobot

This notebook inspects an existing processed store, creates an explicit concatenating plan, converts one episode, and validates the target. Raw GoPro and SLAM processing are outside this workflow. Only the optional second conversion and merge are gated by `RUN_MERGE`.

In [ ]:
from pathlib import Path
from pprint import pprint

from leport import convert, inspect, merge, validate
from leport.conversion.plan import (
    ConversionPlan,
    FeatureMapping,
    FeatureSpec,
    TargetConfig,
    TaskProvider,
    save_plan,
)
from leport.sources import EpisodeSelection

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Project root not found; start from the leport root or notebooks/")

source_path = project_root / "data/umi/cup_in_the_wild.zarr.zip"
output_root = project_root / "outputs"
plan_path = output_root / "umi-cup-demo-a.yaml"
target_root = output_root / "umi-cup-demo-a"

print("Configured processed source:", source_path)

## 1. Inspect the processed source

Inspection validates the UMI replay-buffer signature, derives episode slices from `meta/episode_ends`, and reports array schemas without decoding image payloads.

In [ ]:
# Read the downloaded training-ready store instead of manufacturing notebook data.
if not source_path.exists():
    raise FileNotFoundError(f"Processed UMI source not found: {source_path}")
inspection = inspect(source_path, adapter="umi")
pprint(inspection.to_dict())

## 2. Define explicit conversion semantics

The action order follows the reviewed UMI policy convention, but the plan records it explicitly. Confirm FPS, task text, robot type, coordinate frames, units, and camera choice for this store.

In [ ]:
# Keep every semantic decision visible and reviewable before conversion.
episode_selection = EpisodeSelection(episode_ids=("episode_0",))
fps = 10
task = "arrange the cups"
robot_type = "umi-gripper"
action_sources = (
    "robot0_eef_pos",
    "robot0_eef_rot_axis_angle",
    "robot0_gripper_width",
)
image_source = "camera0_rgb"
image_target = "observation.images.wrist"
use_videos = False

image_field = inspection.field(image_source)
if image_field is None or not image_field.schema_consistent or not image_field.image_candidate:
    raise ValueError(f"Expected one consistent RGB field: {image_source}")
image_shape = image_field.shapes[0]

## 3. Create and save a plan

Both `observation.state` and `action` concatenate the three stored fields in the declared order. The adapter does not synthesize action values.

In [ ]:
# Build the explicit multi-source action mapping supported by ConversionPlan.
features = {
    "observation.state": FeatureSpec("float32", (7,)),
    image_target: FeatureSpec("video" if use_videos else "image", image_shape),
    "action": FeatureSpec("float32", (7,)),
}
mappings = {
    "observation.state": FeatureMapping(action_sources, operation="concat"),
    image_target: FeatureMapping((image_source,)),
    "action": FeatureMapping(action_sources, operation="concat"),
}
plan = ConversionPlan(
    adapter="umi",
    source=source_path,
    selection=episode_selection,
    target=TargetConfig(
        repo_id="local/umi-cup-demo-a",
        root=target_root,
        robot_type=robot_type,
        use_videos=use_videos,
    ),
    fps=fps,
    task=TaskProvider("static", task),
    features=features,
    mappings=mappings,
)
pprint(plan.to_dict())
output_root.mkdir(parents=True, exist_ok=True)
if plan_path.exists():
    raise FileExistsError(f"Plan already exists: {plan_path}")
save_plan(plan, plan_path)

## 4. Convert

Conversion preflights the selected slice, reads only mapped arrays, decodes only current Zarr chunks, and commits after LeRobot reload validation succeeds.

In [ ]:
# Preserve completed datasets; LePort never overwrites a non-empty target.
if target_root.exists():
    raise FileExistsError(f"Target already exists: {target_root}")
conversion_result = convert(plan_path)
pprint(conversion_result.validation.to_dict())

## 5. Validate against the source

The reviewed plan supplies expected episode length, task text, feature schemas, and decoded visual features.

In [ ]:
# Compare the committed target with the exact source selection and plan.
validation_report = validate(target_root, plan=plan_path)
pprint(validation_report.to_dict())

## 6. Merge converted episodes (optional)

Set `RUN_MERGE = True` to convert disjoint `episode_1` with the same schema and merge both completed LeRobot datasets. The processed Zarr store is never a merge input.

In [ ]:
# Keep the second conversion and merge disabled in the default workflow.
RUN_MERGE = False

merge_result = None
if RUN_MERGE:
    second_plan_path = output_root / "umi-cup-demo-b.yaml"
    second_target = output_root / "umi-cup-demo-b"
    merged_target = output_root / "umi-cup-demo-merged"
    second_plan = ConversionPlan(
        adapter="umi",
        source=source_path,
        selection=EpisodeSelection(episode_ids=("episode_1",)),
        target=TargetConfig(
            repo_id="local/umi-cup-demo-b",
            root=second_target,
            robot_type=robot_type,
            use_videos=use_videos,
        ),
        fps=fps,
        task=TaskProvider("static", task),
        features=features,
        mappings=mappings,
    )
    if second_plan_path.exists() or second_target.exists():
        raise FileExistsError("Second plan or target already exists")
    save_plan(second_plan, second_plan_path)
    second_result = convert(second_plan_path)
    pprint(second_result.validation.to_dict())

    if merged_target.exists():
        raise FileExistsError(f"Merge target already exists: {merged_target}")
    merge_result = merge(
        (target_root, second_target),
        target_root=merged_target,
        repo_id="local/umi-cup-demo-merged",
    )
    pprint(merge_result.to_dict())
else:
    print("Optional merge skipped.")

## Equivalent CLI commands

The notebook writes the explicit multi-source action plan because the convenience `leport plan` command accepts one action selector. After reviewing the generated YAML, run these commands from the repository root:

```bash
uv run leport inspect data/umi/cup_in_the_wild.zarr.zip \
  --adapter umi --episode episode_0 --json

uv run leport plan --check outputs/umi-cup-demo-a.yaml --json
uv run leport convert --config outputs/umi-cup-demo-a.yaml --json
uv run leport validate outputs/umi-cup-demo-a \
  --config outputs/umi-cup-demo-a.yaml --json

# Optional: run the Python cell that writes the disjoint episode_1 plan first.
uv run leport convert --config outputs/umi-cup-demo-b.yaml --json
uv run leport merge outputs/umi-cup-demo-a outputs/umi-cup-demo-b \
  --target outputs/umi-cup-demo-merged \
  --repo-id local/umi-cup-demo-merged --json
```

> Review existing plan and target paths before rerunning. LePort never overwrites completed outputs silently.